# scKNIFE synthetic regression checks

All 1,000 cells, counts and group labels are simulated with seed 42. No Bassez expression or patient metadata are included. Gene identifiers come from the unchanged Version 1 graph. These checks exercise software APIs; graph-derived biological labels and activities have no biological interpretation on this fixture. For real data, use the separate 10,000-cell tutorial and obtain inputs from the original source.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import networkx as nx
import scanpy as sc
import matplotlib.pyplot as plt
from IPython.display import display, Image
from scknife import (run_full_pipeline, compute_activity_scores,
    run_h_graph_clustering_fixed, run_celltype_assignment,
    run_cellstate_assignment, h_projection)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "tutorial/data/synthetic_regression_1000.h5ad").exists())
OUT = Path(os.environ.get("SCKNIFE_TUTORIAL_OUTPUT", ROOT / "tutorial/outputs"))
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
G = nx.read_graphml(ROOT / "graphs/version_1/unified_graph.graphml")
raw = sc.read_h5ad(ROOT / "tutorial/data/synthetic_regression_1000.h5ad")
print(raw.shape, G.number_of_nodes(), G.number_of_edges())
assert raw.n_obs == 1000


## Preprocess on a nonnegative scale
Normalize each cell over all available genes to 10,000 counts and apply natural-log `log1p`. Then retain the 1,000 highest-mean genes that exactly match graph node names. This order avoids redefining library size after gene selection. scKNIFE does not normalize input automatically; centered or scaled negative values are unsuitable for NMF. Sparse input is supported but the fitting expression matrix is materialized as dense with a configurable memory guard.

In [ ]:
adata = raw.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
shared = adata.var_names.isin(G.nodes)
adata = adata[:, shared].copy()
means = np.asarray(adata.X.mean(axis=0)).ravel()
top = np.argsort(means, kind="stable")[-1000:]
adata = adata[:, np.sort(top)].copy()
assert adata.n_vars == 1000
print("Prepared cells × genes:", adata.shape)


## Compare two small models
Fit ranks 4 and 8 on the same 200 cells with the same penalties and seed. Report reconstruction error and factor count explicitly. Training reconstruction generally favors larger models, so it is not by itself a defensible selection criterion. We use a predeclared rank of 8 for the subsequent demonstration; biological analysis needs a broader stability and held-out evaluation.

In [ ]:
idx = np.sort(np.random.default_rng(SEED).choice(adata.n_obs, 200, replace=False))
comparison = []
for rank in (4, 8):
    trial = run_full_pipeline(adata[idx].copy(), G, k=rank, lambda_reg=1,
        beta_W=0.01, beta_H=0.01, n_iter=20, tol=0, random_state=SEED, verbose=False)
    H = trial.obs[[f"H_graph_{i}" for i in range(rank)]].to_numpy().T
    X = trial.X.toarray().T if sp.issparse(trial.X) else trial.X.T
    residual = X - trial.uns["Rg"] @ trial.uns["W_graph"] @ H
    comparison.append({"rank": rank, "relative_reconstruction_error":
        np.linalg.norm(residual) / np.linalg.norm(X),
        "final_objective": trial.uns["loss_history"][-1]})
    plt.close("all")
comparison = pd.DataFrame(comparison)
comparison.to_csv(OUT / "rank_comparison.csv", index=False)
display(comparison)


## Fit all bundled cells and cluster H
W has one row per graph node; H has one column per cell. The fixed H-space clustering settings below are illustrative. The existing `synthetic_group` labels do not enter fitting, PCA, neighborhood construction, or Leiden clustering.

In [ ]:
fit = run_full_pipeline(adata.copy(), G, k=8, lambda_reg=1,
    beta_W=0.01, beta_H=0.01, n_iter=60, tol=0, random_state=SEED, verbose=False)
fit, clustering = run_h_graph_clustering_fixed(fit, pca_dims=6,
    n_neighbors=15, resolution=0.5, random_state=SEED)
assert np.isfinite(fit.uns["W_graph"]).all()
assert len(fit.uns["loss_history"]) == 60
clustering.to_csv(OUT / "clustering.csv", index=False)
display(clustering)
sc.pl.umap(fit, color=["leiden_clusters_str", "synthetic_group"], show=False)
plt.savefig(OUT / "umap.png", dpi=120, bbox_inches="tight")
display(Image(filename=str(OUT / "umap.png")))
plt.close("all")


## Exercise activity and legacy scoring APIs
`compute_activity_scores` returns AnnData and stores the **nodes × cells DataFrame** in `uns['activity_scores']`. Factors are aligned by their numeric suffix before multiplying W and H. Activity represents a model-derived score, not measured abundance or metabolic flux. The legacy marker/state scoring calls below are exercised only as software compatibility checks on invented counts. Their returned labels have no biological interpretation and are not used by the main 10,000-cell tutorial.

In [ ]:
fit = compute_activity_scores(fit, batch_size=256)
activity = fit.uns["activity_scores"]
assert activity.shape == (G.number_of_nodes(), fit.n_obs)
assert activity.columns.equals(fit.obs_names)
def nodes_of_kind(kind):
    return [n for n, attrs in G.nodes(data=True) if attrs.get("kind") == kind]
fit, celltype_tables, _ = run_celltype_assignment(fit, activity,
    nodes_of_kind("cell_type"), methods=("highest_as",))
fit, state_table, state_scores = run_cellstate_assignment(fit, activity,
    nodes_of_kind("cell_state"), cluster_key="leiden_clusters")
celltype_tables["highest_as"].to_csv(OUT / "celltype_mapping.csv", index=False)
state_table.to_csv(OUT / "cellstate_mapping.csv", index=False)
display(celltype_tables["highest_as"])
display(state_table)


## Metabolic-task summaries and W-derived factor groups
Summarize task-node activity by cluster. These descriptive averages are not differential tests and do not account for patient effects. For a small factor hierarchy, group columns of W using cosine distances across graph nodes. This grouping uses W directly, without reference cell-type averages.

In [ ]:
tasks = activity.loc[nodes_of_kind("task")]
task_means = tasks.T.groupby(fit.obs["leiden_clusters"], observed=True).mean().T
task_means.to_csv(OUT / "metabolic_task_means.csv")
display(task_means.loc[task_means.mean(axis=1).nlargest(10).index])
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
W = fit.uns["W_graph"]
distance = pdist(W.T, metric="cosine")
assert np.isfinite(distance).all()
factor_groups = fcluster(linkage(distance, method="average"), t=3, criterion="maxclust")
pd.DataFrame({"factor": np.arange(W.shape[1]), "W_group": factor_groups}).to_csv(
    OUT / "W_factor_groups.csv", index=False)


## Project held-out cells
Train a small model on 100 cells and project another 10 cells using the same genes and preprocessing. `h_projection` returns a new AnnData whose X contains reconstructed expression, not the original counts. This example checks the API; ten test cells cannot establish generalization performance.

In [ ]:
reference = adata[:110].copy()
train_idx = np.arange(100)
test_idx = np.arange(100, 110)
trained = run_full_pipeline(reference[train_idx].copy(), G, k=4,
    lambda_reg=1, beta_W=0.01, beta_H=0.01, n_iter=20,
    tol=0, random_state=SEED, verbose=False)
reference.uns["node_names"] = trained.uns["node_names"]
X_test = reference[test_idx].X
X_test = X_test.toarray().T if sp.issparse(X_test) else X_test.T
projected = h_projection(W_graph=trained.uns["W_graph"], Rg=trained.uns["Rg"],
    H_train=trained.obs[[f"H_graph_{i}" for i in range(4)]].to_numpy().T,
    X_test=X_test, train_idx=train_idx, test_idx=test_idx,
    adata_orig=reference, k=4)
assert projected.shape == reference.shape
assert np.isfinite(projected.X).all()
np.save(OUT / "projected_H.npy", projected.uns["H_combined"])
plt.close("all")


## Exercise signaling on synthetic inputs
An invented feature pair exercises the scoring and permutation APIs. It is not a biological ligand–receptor result.


In [ ]:
from scknife.signaling import activity_lr as lr
synthetic = np.random.default_rng(SEED).random((20, 2))
labels = np.repeat([0, 1], 10)
pairs = pd.DataFrame({"ligand_index": [0], "receptor_index": [1],
                      "confidence": [1.0], "cofactor_weight": [1.0]})
means, _ = lr.group_means(synthetic, labels, 2)
specificity = np.ones_like(means)
observed = lr.communication_matrix(means, specificity, pairs)
previous_dir = lr.DATA_DIR
lr.DATA_DIR = OUT
try:
    pvalues, completed = lr.run_permutations(synthetic, labels, observed,
        specificity, pairs, 2, 19, 10, True, "synthetic-tutorial")
finally:
    lr.DATA_DIR = previous_dir
assert completed == 19 and np.all((pvalues >= 1/20) & (pvalues <= 1))
display(pd.DataFrame(pvalues))


## Save and verify
Store cell annotations, activity, and the fitted AnnData. The activity table is kept in a separate parquet file to avoid duplicating it inside the H5AD. A fresh run overwrites these tutorial outputs, not source data.

In [ ]:
activity.to_parquet(OUT / "activity_scores.parquet")
fit.obs.to_csv(OUT / "cell_annotations.csv")
fit.uns.pop("activity_scores", None)
fit.write_h5ad(OUT / "fitted_subset.h5ad")
reloaded = sc.read_h5ad(OUT / "fitted_subset.h5ad")
assert reloaded.shape == fit.shape
assert reloaded.uns["W_graph"].shape == (G.number_of_nodes(), 8)
print("Tutorial completed; results saved to the configured output directory.")
